# Compendium Data Pipeline (Simplified)

Reads the country questionnaires for each chapter (Health, Population,
Education, Labor, Poverty, Housing), reshapes and merges every Excel file's
data/source tables, corrects the labels against `translation dict.xlsx`,
translates them into the other language, and saves per-chapter
`<Chapter>_AR.xlsx` / `<Chapter>_EN.xlsx` outputs to the COMPENDIUM-ARAB
SOCIETY folder.

**Both directions are supported.** The questionnaires are filed by language,
one folder per language:

| Folder | Holds |
| --- | --- |
| `datacollector_received_quest_AR\<Chapter>\` | the Arabic questionnaires |
| `datacollector_received_quest_EN\<Chapter>\` | the English questionnaires |

**`TRANSLATE_TO`** decides the direction, and everything else follows from it -
which folder is read, which dictionary is used, and which file is the
translation. It comes from the `TRANSLATE_TO` environment variable if one is
set, otherwise from `DEFAULT_TRANSLATE_TO` in the config cell:

| `TRANSLATE_TO` | Reads | Dictionary used |
| --- | --- | --- |
| `"EN"` | `..._AR` | `DICTIONARY_AR_TO_EN` |
| `"AR"` | `..._EN` | `DICTIONARY_EN_TO_AR` |

Either way both `<Chapter>_AR.xlsx` and `<Chapter>_EN.xlsx` are written: the
corrected source sheets go to their own language's file, their translations to
the other.

This version favors simplicity over defensiveness: plain functions (no
classes), one try/except per sheet, and flat, plainly-named dictionaries.


In [1]:
"""
CELL: Imports and logging setup.
"""
import difflib
import logging
import os
from collections import defaultdict
from pathlib import Path

import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("compendium_pipeline")


## Config / paths

In [ ]:
"""
CELL: Configuration - paths, chapters, the translation direction, the
fuzzy-match cutoff, and the column names the pipeline creates for itself.
"""
DATA_COLLECTOR_PATH = Path(r"C:\Users\RSHIRINI\OneDrive - United Nations\Desktop\DSS\DATA COLLECTOR")
TRANSLATION_DICT_PATH = DATA_COLLECTOR_PATH / "translation dict.xlsx"
OUTPUT_PATH = Path(r"C:\Users\RSHIRINI\OneDrive - United Nations\Desktop\DSS\COMPENDIUM-ARAB SOCIETY")

# CHAPTERS = ["Health", "Population", "Education", "Labor", "Poverty", "Housing"]
CHAPTERS = ["Population", "Labor"]

# The two languages the pipeline can read and write.
LANGUAGES = ["AR", "EN"]
OTHER_LANGUAGE = {"AR": "EN", "EN": "AR"}

# ---------------------------------------------------------------------------
# WHICH DIRECTION TO RUN IN.
#   "EN" -> read the Arabic questionnaires, translate into English
#   "AR" -> read the English questionnaires, translate into Arabic
#
# The default below is what you get when you just run the notebook. To run
# the other way for one run WITHOUT editing this file, set the TRANSLATE_TO
# environment variable first - the notebook picks it up:
#
#   in a terminal      TRANSLATE_TO=AR jupyter nbconvert --execute <notebook>
#   in a notebook cell os.environ["TRANSLATE_TO"] = "AR"   (before this cell)
#
# This is also how Claude Code runs it when you say "translate to AR" - so
# asking for a one-off run in the other direction never edits your file.
DEFAULT_TRANSLATE_TO = "EN"

TRANSLATE_TO = os.environ.get("TRANSLATE_TO", DEFAULT_TRANSLATE_TO).strip().upper()
if TRANSLATE_TO not in LANGUAGES:
    raise ValueError(
        f"TRANSLATE_TO must be one of {LANGUAGES}, not {TRANSLATE_TO!r}. "
        f"Check the TRANSLATE_TO environment variable."
    )
# ---------------------------------------------------------------------------

# Everything else follows from it. The questionnaires are filed by language,
# one folder each, so the direction also decides which folder is read - and
# load_dictionary() below uses it to pick between the two dictionaries.
TRANSLATE_FROM = OTHER_LANGUAGE[TRANSLATE_TO]
RECEIVED_QUEST_PATH = DATA_COLLECTOR_PATH / f"datacollector_received_quest_{TRANSLATE_FROM}"

# Used only for a sheet whose columns are in neither script (a mangled sheet).
DEFAULT_LANGUAGE = TRANSLATE_FROM

# A fuzzy match must score at least this well (0 to 1) to be used.
# Below this, we leave the value alone and just log a warning, instead of
# guessing and possibly writing something wrong.
FUZZY_MATCH_CUTOFF = 0.6

# Column names the pipeline creates itself, written here in Arabic. The code
# below calls column_name(..., language) to get the right spelling for the
# sheet in hand, so no English spelling has to be typed out a second time.
YEAR_COLUMN = "السنة"
VALUE_COLUMN = "العدد"
CHAPTER_COLUMN = "الفصل"

# Columns used to attach the right source/citation row to each data row:
# year, indicator, country.
MERGE_COLUMNS = ["السنة", "المؤشر", "الدولة"]

# Columns that are TRANSLATED normally but never FUZZY-matched. Source
# holds free-text citations (survey and census names). An exact dictionary
# lookup on them is safe and wanted; guessing from the nearest entry is not,
# because two citations differing by one digit ("...2021" / "...2022") score
# high enough to overwrite each other. Citations the dictionary has no entry
# for pass through unchanged and are picked up by export_untranslated().
COLUMNS_NOT_FUZZY_MATCHED = ["المصدر"]


## Load the translation dictionary

Reads `translation dict.xlsx` once and builds **two dictionaries** from it:

| Name | Maps |
| --- | --- |
| `DICTIONARY_AR_TO_EN` | Arabic column names and values -> English |
| `DICTIONARY_EN_TO_AR` | English column names and values -> Arabic |

Each one is a `(column_map, value_map)` pair. `TRANSLATE_TO` decides which is
used for a run: translating into English uses `DICTIONARY_AR_TO_EN`, into
Arabic uses `DICTIONARY_EN_TO_AR`.

The same two dictionaries also serve as each language's **vocabulary** - the
set of known column names and values that `correct_with_dictionary()` matches
misspellings against.


In [3]:
"""
CELL: Load translation dict.xlsx into the two dictionaries, one per direction.
"""


def load_dictionary():
    """Builds the column and value lookups for both directions.

    Each direction is a pair (column_map, value_map):
        column_map = {source column name: target column name}
        value_map  = {source column name: {source value: target value}}

    Both come from the same file - the direction only decides which of its
    columns is read as "from" and which as "to".

    The KEYS of each direction's maps double as that language's vocabulary:
    the column names and values the correction step matches against.
    """
    dict_df = pd.read_excel(TRANSLATION_DICT_PATH, engine="openpyxl")

    translations = {}
    for source, target in [("AR", "EN"), ("EN", "AR")]:
        # The file's headers are col_ar / col_en and val_ar / val_en.
        source_column_header = f"col_{source.lower()}"
        target_column_header = f"col_{target.lower()}"
        source_value_header = f"val_{source.lower()}"
        target_value_header = f"val_{target.lower()}"

        column_map = {}  # source column name -> target column name
        value_map = {}   # source column name -> {source value: target value}

        for source_column in dict_df[source_column_header].dropna().unique():
            rows = dict_df[dict_df[source_column_header] == source_column]
            column_map[source_column] = rows[target_column_header].iloc[0]
            value_map[source_column] = {
                value_from: value_to
                for value_from, value_to
                in zip(rows[source_value_header], rows[target_value_header])
                if pd.notna(value_from)
            }

        translations[(source, target)] = (column_map, value_map)

    # The dictionary also has a "Chapter" column (col_en == "Chapter") whose
    # rows are exactly the chapter names, e.g. "Labor" <-> "عمالة".
    chapter_rows = dict_df[dict_df["col_en"] == "Chapter"]
    chapter_to_arabic = dict(zip(chapter_rows["val_en"], chapter_rows["val_ar"]))

    return translations, chapter_to_arabic


TRANSLATIONS, CHAPTER_TO_ARABIC = load_dictionary()

# The two dictionaries, under plain names. Each is a (column_map, value_map)
# pair. A run with TRANSLATE_TO = "EN" uses DICTIONARY_AR_TO_EN; "AR" uses
# DICTIONARY_EN_TO_AR. translate() picks between them by direction.
DICTIONARY_AR_TO_EN = TRANSLATIONS[("AR", "EN")]
DICTIONARY_EN_TO_AR = TRANSLATIONS[("EN", "AR")]


def vocabulary(language):
    """That language's known column names and known values, as the pair
    (column_map, value_map) - i.e. the dictionary that translates OUT of it.
    Used both to translate and to fuzzy-match against."""
    return DICTIONARY_AR_TO_EN if language == "AR" else DICTIONARY_EN_TO_AR


def column_name(arabic_name, language):
    """One of the pipeline's own column names, spelled for the given language:
    the Arabic name unchanged for "AR", its dictionary translation for "EN".
    Looking the English spelling up here (rather than typing it a second time)
    keeps it from ever drifting apart from what translate() produces."""
    arabic_to_english, _ = DICTIONARY_AR_TO_EN
    return arabic_name if language == "AR" else arabic_to_english[arabic_name]


arabic_columns, _ = DICTIONARY_AR_TO_EN
logger.info(
    f"Dictionary loaded: {len(arabic_columns)} column names, "
    f"translating {TRANSLATE_FROM} -> {TRANSLATE_TO} "
    f"(using DICTIONARY_{TRANSLATE_FROM}_TO_{TRANSLATE_TO})"
)


13:52:08 | INFO | Dictionary loaded: 27 Arabic columns available


## `extract_tables()`

Each sheet has two tables marked by an `index` column: `index=1` is the data
table, `index=2` is the source table. A row where column 0 says `"index"`
holds the column names for the table that follows it.

In [ ]:
"""
CELL: extract_tables() - split one raw sheet into its data table and source table.
"""


def extract_tables(raw_sheet):
    header_rows = raw_sheet.index[raw_sheet[0] == "index"].tolist()
    data_header_row, source_header_row = header_rows[0], header_rows[1]

    # .str.strip() matters here: a stray trailing space in a raw header (seen
    # in a couple of Morocco files) would otherwise silently break the merge
    # key match in reshape_and_merge() and cause a duplicate-column crash in
    # correct_with_dictionary().
    data_columns = raw_sheet.iloc[data_header_row].dropna().str.strip()
    data_table = raw_sheet[raw_sheet[0] == "1"][data_columns.index].copy()
    data_table.columns = data_columns.values
    data_table = data_table.drop(columns=["index"])

    source_columns = raw_sheet.iloc[source_header_row].dropna().str.strip()
    source_table = raw_sheet[raw_sheet[0] == "2"][source_columns.index].copy()
    source_table.columns = source_columns.values
    source_table = source_table.drop(columns=["index"])

    return data_table, source_table


## `detect_language()`

Works out whether a sheet is written in Arabic or English **by looking at the
script its column names are written in** - Arabic letters occupy their own
Unicode block, Latin letters another. No dictionary needed, so it still works
on a column the dictionary has never seen.

The pipeline does not use this to decide what to do - `TRANSLATE_TO` and the
folder do that. It is used as a check: if a sheet in the Arabic folder turns
out to be English, you get a warning naming the file and sheet instead of a
silent mess of untranslated rows.


In [ ]:
"""
CELL: detect_language() - is this sheet written in Arabic or English?
"""


def looks_arabic(text):
    """True if the text contains at least one Arabic letter. U+0600-U+06FF is
    the Arabic Unicode block; English text has nothing in it."""
    return any("\u0600" <= character <= "\u06ff" for character in str(text))


def detect_language(table):
    """Returns "AR" or "EN" based on the script the column names are written in.

    Column names are a far better signal than cell values: there are only a
    couple of dozen of them, while values include numbers, codes and
    country-specific free text that can be in either script.

    Falls back to DEFAULT_LANGUAGE if the table has no usable column names.
    """
    names = [str(c) for c in table.columns if not str(c).isdigit()]
    if not names:
        return DEFAULT_LANGUAGE

    arabic_names = sum(1 for name in names if looks_arabic(name))
    return "AR" if arabic_names > len(names) / 2 else "EN"


## `reshape_and_merge()`

Unpivots the data table's year columns (any column whose name is all
digits) into two columns (year, value), then merges in the matching row
from the source table so every data point carries its source.

Takes the sheet's `language` so the two columns it creates, and the columns
it merges on, are named in that same language.


In [5]:
"""
CELL: reshape_and_merge() - wide-to-long reshape, then attach the source table.
"""


def reshape_and_merge(data_table, source_table, language):
    id_columns = [c for c in data_table.columns if not str(c).isdigit()]
    year_columns = [c for c in data_table.columns if str(c).isdigit()]

    long_table = data_table.melt(
        id_vars=id_columns,
        value_vars=year_columns,
        var_name=column_name(YEAR_COLUMN, language),
        value_name=column_name(VALUE_COLUMN, language),
    )

    # Merge on year/indicator/country spelled in the sheet's own language,
    # and only on the ones both tables actually have.
    merge_columns = [column_name(c, language) for c in MERGE_COLUMNS]
    merge_columns = [c for c in merge_columns if c in long_table.columns and c in source_table.columns]
    return pd.merge(long_table, source_table, on=merge_columns, how="left")


## `correct_with_dictionary()`

Works in whichever language the sheet is written in - the `language` argument
picks which vocabulary to match against, so an English sheet is checked
against the English column names and values, an Arabic one against the Arabic.

For every column: if its name is already a known column name in that language,
leave it. Otherwise, find the known column name it's most similar to, and
rename it there - but only if that similarity score clears
`FUZZY_MATCH_CUTOFF`.

Then do the same thing for every value within each known column: if the
value is already known, leave it; otherwise replace it with the closest
known value, if close enough. Every actual change is printed.

Values in the **Source** column are never fuzzy-matched (see
`COLUMNS_NOT_FUZZY_MATCHED`) - two citations differing by a single digit
score high enough to overwrite each other, so the nearest entry would be a
wrong answer rather than a correction. They are still *translated* by exact
lookup. A misspelled Source *header* is still fixed, which is what folds
Iraq's `المصادر` back into the Source column.


In [6]:
"""
CELL: correct_with_dictionary() - fix column names and cell values, in either language.
"""


def best_match(text, choices):
    """Compares text against every choice and returns (best_choice, score) -
    the one difflib considers most similar, and how similar (0 to 1)."""
    best_choice, best_score = None, -1
    for choice in choices:
        score = difflib.SequenceMatcher(None, str(text), str(choice)).ratio()
        if score > best_score:
            best_choice, best_score = choice, score
    return best_choice, best_score


def correct_with_dictionary(table, language, chapter, file_name, sheet_name):
    """Fixes misspelled column names and values by matching them against the
    dictionary's vocabulary for `language` ("AR" or "EN")."""
    known_columns, known_values_by_column = vocabulary(language)
    never_guessed = [column_name(c, language) for c in COLUMNS_NOT_FUZZY_MATCHED]

    table = table.copy()
    replacements = 0
    where = f"[{chapter}/{language}] {file_name} | {sheet_name}"

    # 1. Fix column names.
    for column in list(table.columns):
        if column in known_columns:
            continue  # already a known column name, nothing to fix

        match, score = best_match(column, known_columns.keys())
        if match is not None and score >= FUZZY_MATCH_CUTOFF:
            print(f"{where} | column: {column} -> {match} (score={score:.2f})")
            table = table.rename(columns={column: match})
            replacements += 1
        elif match is not None:
            logger.warning(
                f"{where} | column '{column}' has no good match "
                f"(closest is '{match}', score={score:.2f}) - left unchanged"
            )

    # 2. Fix cell values, one known column at a time.
    for column in table.columns:
        if column in never_guessed:
            continue  # free-text citations - translated, but never guessed at

        known_values = known_values_by_column.get(column)
        if not known_values:
            continue  # not a dictionary column, or it has no fixed vocabulary (e.g. Year, Value)

        for value in table[column].dropna().unique():
            if value in known_values:
                continue  # already a known value, nothing to fix

            match, score = best_match(value, known_values.keys())
            if match is not None and score >= FUZZY_MATCH_CUTOFF:
                print(f"{where} | {column}: {value} -> {match} (score={score:.2f})")
                table[column] = table[column].replace(value, match)
                replacements += 1
            elif match is not None:
                logger.warning(
                    f"{where} | {column} value '{value}' has no good match "
                    f"(closest is '{match}', score={score:.2f}) - left unchanged"
                )

    return table, replacements


## `translate()`

Swaps every value for its equivalent in the other language, then renames the
column - using the dictionaries built above.

Choose the direction with `source` and `target`:

| Call | Direction |
| --- | --- |
| `translate(table)` | Arabic to English (the default, what the pipeline uses) |
| `translate(table, source="AR", target="EN")` | Arabic to English, spelled out |
| `translate(table, source="EN", target="AR")` | English to Arabic |

Anything the dictionary has no entry for is left exactly as it is.

**Source is translated like everything else** - by exact dictionary lookup.
Citations the dictionary has not been taught yet pass through unchanged and
are collected by `export_untranslated()` further down.


In [7]:
"""
CELL: translate() - swap one language for the other, in whichever direction you ask for.
"""


def translate(table, source="AR", target="EN"):
    """Translate a table's cell values and column names from one language to
    the other.

    source and target are "AR" or "EN":
        translate(table)                            # Arabic  -> English
        translate(table, source="EN", target="AR")  # English -> Arabic

    Values are replaced first and the column renamed second, because the value
    lookup is keyed by the column's ORIGINAL name - renaming first would lose
    it. Anything the dictionary has no entry for - including a Source citation
    it has not been taught yet - is left exactly as it is, and shows up in
    export_untranslated() below as a gap to fill.
    """
    if (source, target) not in TRANSLATIONS:
        raise ValueError(f"Can't translate {source} -> {target}. Available: {list(TRANSLATIONS)}")

    column_map, value_map = TRANSLATIONS[(source, target)]

    table = table.copy()
    for column in list(table.columns):
        if column in value_map:
            table[column] = table[column].replace(value_map[column])
        if column in column_map:
            table = table.rename(columns={column: column_map[column]})
    return table


## Error tracking

`log_failure()` is the one place that logs a failure and remembers it, so
the final cell can print one consolidated summary grouped by chapter.

In [8]:
"""
CELL: Error tracking - log a failure and remember it for the run summary.
"""
FAILURES = defaultdict(list)  # chapter -> list of {file, sheet, step, error}

# Plain-language guess at what's wrong, keyed by which step failed.
LIKELY_CAUSES = {
    "read": "file may be corrupted, password-protected, or not a valid .xlsx",
    "extract": "sheet may be missing an 'index' column, or index values are not 1/2 as expected",
    "detect language": "sheet's column names match neither the Arabic nor the English dictionary",
    "reshape_and_merge": "data table may be missing year columns, or the merge columns don't match the source table",
    "dictionary correction": "column names or values may be malformed and unable to be matched",
    "translation": "a column or value may not have a corresponding entry in the other language",
}


def log_failure(chapter, file_name, sheet_name, step, error):
    likely_cause = LIKELY_CAUSES.get(step, "unexpected error, inspect the sheet manually")
    logger.error(
        f"[ERROR] Chapter={chapter} | File={file_name} | Sheet={sheet_name} | "
        f"Step={step} | {type(error).__name__}: {error} | Likely cause: {likely_cause}"
    )
    FAILURES[chapter].append({
        "file": file_name,
        "sheet": sheet_name,
        "step": step,
        "error": f"{type(error).__name__}: {error}",
    })


## `process_chapter()`

Runs every step, for every sheet, for every file in one chapter. Each sheet
is processed inside one try/except; a plain `step` variable tracks which
stage we're at, so a failure is logged with the right step name without
needing a separate try/except per step.

The sheet's language is `TRANSLATE_FROM` - it comes from which folder the run
is reading, not from the sheet itself. `detect_language()` is still called on
each sheet, purely to warn you if a file has been filed in the wrong folder.

Every sheet contributes to **both** output files: the corrected sheet goes to
its own language's file, and its translation to the other. So one run always
produces a complete `<Chapter>_AR.xlsx` and `<Chapter>_EN.xlsx`.


In [9]:
"""
CELL: process_chapter() - ties every step together for one chapter, and saves the result.
"""


def process_chapter(chapter):
    folder = RECEIVED_QUEST_PATH / chapter
    if not folder.exists():
        logger.warning(f"Chapter folder not found, skipping: {folder}")
        return

    files = sorted(f for f in folder.glob("*.xlsx") if not f.name.startswith("~$"))
    logger.info(f"Processing {chapter}: {len(files)} file(s) found in {RECEIVED_QUEST_PATH.name}")

    # The language is fixed for the whole run by which folder we are reading.
    language = TRANSLATE_FROM
    other_language = TRANSLATE_TO

    # One list of tables per language. Every sheet adds a table to BOTH: its
    # own corrected version to its own language, its translation to the other.
    tables = {lang: [] for lang in LANGUAGES}
    sheets_done = 0
    wrong_folder = 0
    replacements_made = 0

    for file_path in files:
        file_name = file_path.name
        try:
            xls = pd.ExcelFile(file_path, engine="openpyxl")
        except Exception as error:
            log_failure(chapter, file_name, "-", "read", error)
            continue

        logger.info(f"  {chapter}/{file_name}: {len(xls.sheet_names)} sheet(s)")

        for sheet_name in xls.sheet_names:
            step = "read"
            try:
                raw_sheet = pd.read_excel(xls, sheet_name=sheet_name, header=None, dtype=str)

                step = "extract"
                data_table, source_table = extract_tables(raw_sheet)

                step = "detect language"
                # Only a check - the folder decides. A mismatch means a file
                # has been dropped in the wrong folder, which would otherwise
                # show up as a whole sheet of untranslated rows.
                if detect_language(data_table) != language:
                    wrong_folder += 1
                    logger.warning(
                        f"  {file_name} | {sheet_name}: looks like "
                        f"{detect_language(data_table)} but sits in the {language} "
                        f"folder - processing it as {language} anyway"
                    )

                # Tag every row with its chapter, named in the sheet's language.
                chapter_in_language = chapter if language == "EN" else CHAPTER_TO_ARABIC[chapter]
                data_table[column_name(CHAPTER_COLUMN, language)] = chapter_in_language

                step = "reshape_and_merge"
                merged_table = reshape_and_merge(data_table, source_table, language)

                step = "dictionary correction"
                corrected_table, n = correct_with_dictionary(
                    merged_table, language, chapter, file_name, sheet_name
                )
                replacements_made += n

                step = "translation"
                translated_table = translate(corrected_table, source=language, target=other_language)

            except Exception as error:
                log_failure(chapter, file_name, sheet_name, step, error)
                continue

            tables[language].append(corrected_table)
            tables[other_language].append(translated_table)
            sheets_done += 1

    logger.info(
        f"{chapter}: {sheets_done} sheet(s) processed successfully, "
        f"{replacements_made} dictionary replacement(s) made"
        + (f", {wrong_folder} sheet(s) looked like the wrong language" if wrong_folder else "")
    )

    if not sheets_done:
        logger.warning(f"{chapter}: no data extracted, no output files written")
        return

    # Stack every sheet's rows on top of each other (concatenate, not merge side-by-side).
    for lang in LANGUAGES:
        result = pd.concat(tables[lang], ignore_index=True)
        result.to_excel(OUTPUT_PATH / f"{chapter}_{lang}.xlsx", index=False, engine="openpyxl")
        logger.info(f"{chapter}: saved {chapter}_{lang}.xlsx ({len(result):,} rows)")


## `export_all_values()` - every value in both languages

Writes one Excel file listing every distinct value the pipeline produced, with
the Arabic and the English side by side. Its columns match
`translation dict.xlsx` (`col_ar` / `val_ar` / `col_en` / `val_en`), so any row
you approve can be pasted straight back into the dictionary.

The `translated` column says `no` where the Arabic and English text came out
identical - which is exactly the list of values the dictionary could not
translate, sorted to the top of the file for review.


In [ ]:
"""
CELL: export_all_values() - every distinct value, Arabic and English side by side.
"""


def export_all_values(chapters=CHAPTERS, file_name="all_values_AR_EN.xlsx"):
    """Reads back the <Chapter>_AR.xlsx / <Chapter>_EN.xlsx this pipeline wrote
    and pairs them up column by column, so each distinct value appears once with
    both its spellings.

    The two files are row-for-row translations of each other, so pairing them is
    just reading the same row from each. Year and Value are skipped - they hold
    numbers, not vocabulary.
    """
    rows = []
    year_en = column_name(YEAR_COLUMN, "EN")
    value_en = column_name(VALUE_COLUMN, "EN")

    for chapter in chapters:
        arabic_path = OUTPUT_PATH / f"{chapter}_AR.xlsx"
        english_path = OUTPUT_PATH / f"{chapter}_EN.xlsx"
        if not (arabic_path.exists() and english_path.exists()):
            logger.warning(f"{chapter}: output files not found, skipping")
            continue

        arabic_table = pd.read_excel(arabic_path, engine="openpyxl")
        english_table = pd.read_excel(english_path, engine="openpyxl")

        for arabic_column, english_column in zip(arabic_table.columns, english_table.columns):
            if english_column in (year_en, value_en):
                continue

            pairs = pd.DataFrame({
                "val_ar": arabic_table[arabic_column],
                "val_en": english_table[english_column],
            }).dropna()

            for (arabic_value, english_value), count in pairs.groupby(["val_ar", "val_en"]).size().items():
                same = str(arabic_value).strip() == str(english_value).strip()
                rows.append({
                    "chapter": chapter,
                    "col_ar": arabic_column,
                    "col_en": english_column,
                    "val_ar": arabic_value,
                    "val_en": english_value,
                    "translated": "no" if same else "yes",
                    "rows": count,
                })

    if not rows:
        logger.warning("No values found - run the pipeline first.")
        return pd.DataFrame()

    all_values = pd.DataFrame(rows).drop_duplicates(subset=["col_en", "val_ar", "val_en"])
    # Untranslated first - that is the list worth reviewing.
    all_values = all_values.sort_values(["translated", "col_en", "val_ar"])

    path = OUTPUT_PATH / file_name
    all_values.to_excel(path, index=False, engine="openpyxl")
    untranslated = (all_values["translated"] == "no").sum()
    logger.info(
        f"Saved {path.name}: {len(all_values):,} distinct value(s), "
        f"{untranslated:,} still untranslated"
    )
    return all_values


## Filling the gaps: `export_untranslated()` -> Claude Code -> `update_dictionary()`

The dictionary is the source of truth, and it always wins. But it does not yet
know every value - Source citations especially, since each country writes its
own survey names. A value the dictionary has no entry for passes through
translation unchanged, which is exactly how it is spotted.

The loop is:

1. Run the pipeline.
2. `export_untranslated()` - writes every value that came through untranslated
   to `untranslated_values.xlsx`, with the target-language column blank. Values
   already written in the target language are not listed: an Arabic
   questionnaire citing "MICS 2022" or a URL is already correct.
3. **Ask Claude Code to fill it in.** It reads the file, translates the blank
   column, and calls `update_dictionary()`.
4. `update_dictionary()` appends those rows to `translation dict.xlsx`
   (after taking a timestamped backup).
5. Run the pipeline again - the new entries are now ordinary dictionary hits.

Because the translations land in the dictionary rather than being produced
fresh each run, the pipeline stays **deterministic and reviewable**: the same
input always gives the same output, and every machine translation is a row you
can inspect, correct, or delete.


In [ ]:
"""
CELL: export_untranslated() and update_dictionary() - the gap-filling loop.
"""


def find_untranslated(chapters=CHAPTERS):
    """Every distinct value the dictionary could not translate.

    A value with no dictionary entry is copied through unchanged, so the source
    and target files hold identical text for it - that identity IS the gap.

    Two kinds of identical pair are NOT gaps and are filtered out:
      - Year and Value, which hold numbers rather than vocabulary.
      - A value already written in the target language. Plenty of Arabic
        questionnaires cite their source in English already ("2010 Census",
        "MICS 2022", a URL); those are correct as they stand and there is
        nothing to translate.
    """
    source_language, target_language = TRANSLATE_FROM, TRANSLATE_TO
    year_en = column_name(YEAR_COLUMN, "EN")
    value_en = column_name(VALUE_COLUMN, "EN")

    gaps = []
    for chapter in chapters:
        source_path = OUTPUT_PATH / f"{chapter}_{source_language}.xlsx"
        target_path = OUTPUT_PATH / f"{chapter}_{target_language}.xlsx"
        if not (source_path.exists() and target_path.exists()):
            logger.warning(f"{chapter}: output files not found, skipping")
            continue

        source_table = pd.read_excel(source_path, engine="openpyxl")
        target_table = pd.read_excel(target_path, engine="openpyxl")

        for source_column, target_column in zip(source_table.columns, target_table.columns):
            if target_column in (year_en, value_en) or source_column in (year_en, value_en):
                continue

            pairs = pd.DataFrame({
                "source_value": source_table[source_column],
                "target_value": target_table[target_column],
            }).dropna()

            for (source_value, target_value), count in pairs.groupby(
                ["source_value", "target_value"]
            ).size().items():
                if str(source_value).strip() != str(target_value).strip():
                    continue  # translated fine
                if looks_arabic(source_value) == (target_language == "AR"):
                    continue  # already written in the target language
                gaps.append({
                    "chapter": chapter,
                    f"col_{source_language.lower()}": source_column,
                    f"col_{target_language.lower()}": target_column,
                    f"val_{source_language.lower()}": source_value,
                    f"val_{target_language.lower()}": None,   # <- to be filled in
                    "rows": count,
                })

    if not gaps:
        return pd.DataFrame()

    columns = ["chapter", "col_ar", "col_en", "val_ar", "val_en", "rows"]
    return (
        pd.DataFrame(gaps)
        .reindex(columns=columns)
        .drop_duplicates(subset=["col_ar", "col_en", "val_ar", "val_en"])
        .sort_values(["col_en", "val_ar"])
        .reset_index(drop=True)
    )


def export_untranslated(chapters=CHAPTERS, file_name="untranslated_values.xlsx"):
    """Writes the gaps to an Excel file shaped like translation dict.xlsx, with
    the target-language value column left blank for Claude Code to fill in."""
    gaps = find_untranslated(chapters)
    if gaps.empty:
        logger.info("Nothing untranslated - the dictionary covers every value.")
        return gaps

    path = OUTPUT_PATH / file_name
    gaps.to_excel(path, index=False, engine="openpyxl")
    blank_column = f"val_{TRANSLATE_TO.lower()}"
    logger.info(
        f"Saved {path.name}: {len(gaps):,} value(s) the dictionary could not "
        f"translate. Fill in the '{blank_column}' column, then call update_dictionary()."
    )
    return gaps


def update_dictionary(filled, backup=True):
    """Appends reviewed translations to translation dict.xlsx.

    `filled` is the exported table with the blank value column filled in (a
    DataFrame, or a path to the saved Excel file). Rows missing either side are
    skipped, and a (column, value) pair the dictionary already has is left
    alone - so running this twice changes nothing the second time.

    A timestamped backup is written first, because this edits the project's
    source of truth.
    """
    if not isinstance(filled, pd.DataFrame):
        filled = pd.read_excel(filled, engine="openpyxl")

    needed = ["col_ar", "val_ar", "col_en", "val_en"]
    missing = [c for c in needed if c not in filled.columns]
    if missing:
        raise ValueError(f"missing column(s) {missing}; expected {needed}")

    new_rows = filled[needed].dropna()
    new_rows = new_rows[
        (new_rows["val_ar"].astype(str).str.strip() != "")
        & (new_rows["val_en"].astype(str).str.strip() != "")
    ]
    if new_rows.empty:
        logger.warning("No completed rows to add - is the blank column filled in?")
        return None

    dictionary = pd.read_excel(TRANSLATION_DICT_PATH, engine="openpyxl")
    already_there = set(zip(dictionary["col_ar"], dictionary["val_ar"]))
    to_add = new_rows[
        ~new_rows.apply(lambda r: (r["col_ar"], r["val_ar"]) in already_there, axis=1)
    ]
    if to_add.empty:
        logger.info("Every row is already in the dictionary - nothing to add.")
        return dictionary

    if backup:
        stamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        backup_path = TRANSLATION_DICT_PATH.with_name(
            f"{TRANSLATION_DICT_PATH.stem} backup {stamp}.xlsx"
        )
        dictionary.to_excel(backup_path, index=False, engine="openpyxl")
        logger.info(f"Backed up the dictionary to {backup_path.name}")

    updated = pd.concat([dictionary, to_add.reindex(columns=dictionary.columns)],
                        ignore_index=True)
    updated.to_excel(TRANSLATION_DICT_PATH, index=False, engine="openpyxl")
    logger.info(
        f"Added {len(to_add):,} row(s) to {TRANSLATION_DICT_PATH.name} "
        f"({len(dictionary):,} -> {len(updated):,}). Re-run the pipeline to use them."
    )
    return updated


## Run the pipeline

Reads the folder for `TRANSLATE_FROM` and writes both language files for every
chapter. Change `TRANSLATE_TO` in the config cell to run the other way.


In [ ]:
"""
CELL: Main run - processes every chapter and writes the AR/EN files.

Prints the direction it is running in, then a simple progress bar before each
chapter starts, so you can see how far through the run you are without reading
the detailed log lines. Ends by listing anything the dictionary could not
translate.
"""
FAILURES.clear()

source_of_setting = "environment" if "TRANSLATE_TO" in os.environ else "notebook default"
print(f"Translating {TRANSLATE_FROM} -> {TRANSLATE_TO}  (from the {source_of_setting})")
print(f"Reading from: {RECEIVED_QUEST_PATH}")
print(f"Writing to  : {OUTPUT_PATH}\n")

total_chapters = len(CHAPTERS)
for i, chapter in enumerate(CHAPTERS, start=1):
    bar = "#" * i + "-" * (total_chapters - i)
    print(f"[{bar}] chapter {i}/{total_chapters}: {chapter}")
    process_chapter(chapter)

# Anything the dictionary had no entry for was copied through untranslated.
# Report it here, so it is obvious straight after a run rather than something
# you notice later while reading the output in Excel.
gaps = find_untranslated()
print()
if gaps.empty:
    print("Every value was translated - the dictionary covered all of them.")
else:
    print(f"{len(gaps)} value(s) had NO dictionary entry and were left untranslated:")
    for _, row in gaps.head(10).iterrows():
        print(f"   {row['col_en']}: {row['val_ar']}")
    if len(gaps) > 10:
        print(f"   ... and {len(gaps) - 10} more")
    print("\nTo fix: run export_untranslated(), ask Claude Code to fill in the")
    print("blank column, then update_dictionary() - and run this cell again.")


## Run summary - failures grouped by chapter

In [11]:
"""
CELL: Run summary - every failure from the run above, grouped by chapter.
"""
print("\n" + "=" * 70)
print("RUN SUMMARY - FAILURES BY CHAPTER")
print("=" * 70)

if not FAILURES:
    print("No failures. All files/sheets processed successfully.")
else:
    total = sum(len(v) for v in FAILURES.values())
    print(f"{total} failure(s) across {len(FAILURES)} chapter(s):\n")
    for chapter, failures in FAILURES.items():
        print(f"{chapter} ({len(failures)} failure(s)):")
        for f in failures:
            print(f"  - {f['file']} | Sheet={f['sheet']} | Step={f['step']} | {f['error']}")
        print()



RUN SUMMARY - FAILURES BY CHAPTER
1 failure(s) across 1 chapter(s):

Poverty (1 failure(s)):
  - Qatar poverty.xlsx | Sheet=Poverty_4 | Step=extract | IndexError: list index out of range



## Reports

Both read back the files the run just wrote.

- `export_all_values()` - every distinct value in both languages, to
  `all_values_AR_EN.xlsx`.
- `export_untranslated()` - just the gaps, to `untranslated_values.xlsx`, with
  the target-language column blank for Claude Code to fill in.


In [ ]:
"""
CELL: Write the two report files for the run above.
"""
ALL_VALUES = export_all_values()
UNTRANSLATED = export_untranslated()

UNTRANSLATED.head(15) if not UNTRANSLATED.empty else ALL_VALUES.head(15)
